## Julian Canales
### Notebook 02 Collect Buffer

This notebook uses the trained DQN model that was accomplished in notebook 01 and uses it to collect buffer transition data that will be used in notebook 03.

In [1]:
!pip install -q highway-env stable-baselines3[extra]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 11.0 MB/s eta 0:00:00


In [2]:
import gymnasium as gym, highway_env, numpy as np, random, torch, cloudpickle as pkl
from stable_baselines3 import DQN
from tqdm.auto import trange

In [3]:
SEED, N_TRANS = 123, 1_000
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)

model = DQN.load("roundabout_dqn.zip", device="cuda", print_system_info=False)
model.policy.eval()
config = {
    "observation": {
        "type": "Kinematics",
        "absolute": True,
        "features_range": {
            "x": [-100, 100],
            "y": [-100, 100],
            "vx": [-15, 15],
            "vy": [-15, 15],
        },
    },
    "action": {
        "type": "DiscreteMetaAction",
        "target_speeds": [0, 8, 16],
    },
    "incoming_vehicle_destination": None,
    "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
    "collision_reward": -1,
    "high_speed_reward": 0.2,
    "right_lane_reward": 0,
    "lane_change_reward": -0.05,
    "screen_width": 600,
    "screen_height": 600,
    "centering_position": [0.5, 0.6],
    "duration": 11,
    "normalize_reward": True,
}
env = gym.make("roundabout-v0", config=config)

In [4]:
buffer = []
obs, _ = env.reset(seed=SEED)
for _ in trange(N_TRANS, desc="collect DQN transitions"):
    with torch.no_grad():
        action, _ = model.predict(obs, deterministic=True)
    next_obs, _, term, trunc, _ = env.step(action)
    buffer.append((obs, int(action), next_obs))
    obs = next_obs if not (term or trunc) else env.reset(seed=SEED)[0]

env.close()
with open("transition_buffer.pkl", "wb") as f:
    pkl.dump(buffer, f)

print(f"✓ {len(buffer):,} transitions saved to transition_buffer.pkl")

collect DQN transitions:   0%|          | 0/1000 [00:00<?, ?it/s]

✓ 1,000 transitions saved to transition_buffer.pkl
